# Chapter 1 Workbook (Semester 2026B): Foundations and Error Analysis

This notebook follows the spirit of Chapter 1 (Burden & Burden):
- numerical error and significant digits
- floating-point behavior
- stability and cancellation
- algorithmic thinking with simple computational experiments

Run each example, then complete the exercise cells marked `TODO`.

In [34]:
import math
import random
from decimal import Decimal, getcontext

import matplotlib.pyplot as plt
import numpy as np

getcontext().prec = 50  # High precision reference for selected comparisons

## §1.1 Taylor's Theorem and Polynomial Approximation

The Taylor polynomial $P_n(x)$ approximates a smooth function $f$ near a point $x_0$:

$$P_n(x) = \sum_{k=0}^{n} \frac{f^{(k)}(x_0)}{k!}(x - x_0)^k$$

The **Lagrange remainder** gives us an exact error bound:

$$f(x) = P_n(x) + \frac{f^{(n+1)}(\xi)}{(n+1)!}(x-x_0)^{n+1}$$

for some $\xi$ between $x$ and $x_0$.  This is the key tool Burden uses to derive and **bound truncation error** throughout the book.

In [ ]:
def taylor_sin(x, terms):
    """Approximate sin(x) using the first `terms` non-zero Maclaurin terms."""
    approx = 0.0
    for k in range(terms):
        approx += ((-1) ** k) * x**(2*k + 1) / math.factorial(2*k + 1)
    return approx


x_test = math.pi / 4
true_val = math.sin(x_test)
print(f"True sin(\u03c0/4) = {true_val:.15f}")
print()
print(f"{'terms':>6}  {'approximation':>20}  {'abs error':>15}  {'bound O(x^(2n+3)/(2n+3)!)':>26}")
for n in range(1, 8):
    approx = taylor_sin(x_test, n)
    ae = abs(true_val - approx)
    bound = abs(x_test)**(2*n + 1) / math.factorial(2*n + 1)
    print(f"{n:>6}  {approx:>20.15f}  {ae:>15.8e}  {bound:>26.8e}")

## 1) Absolute and Relative Error

For true value $p$ and approximation $p^*$:
- Absolute error: $|p - p^*|$
- Relative error: $|p - p^*| / |p|$ (if $p \neq 0$)

In [ ]:
def absolute_error(true_value: float, approx_value: float) -> float:
    return abs(true_value - approx_value)


def relative_error(true_value: float, approx_value: float) -> float:
    if true_value == 0:
        raise ValueError("Relative error is undefined when true value is 0.")
    return abs(true_value - approx_value) / abs(true_value)


true_pi = math.pi
approximations = [22 / 7, 355 / 113]

for a in approximations:
    print(f"approx = {a:.12f}")
    print(f"  abs error = {absolute_error(true_pi, a):.12e}")
    print(f"  rel error = {relative_error(true_pi, a):.12e}")

### Exercise 1
Compute the absolute and relative errors for these approximations of $\sqrt{2}$:
- $1.4$
- $1.41$
- $1.414$

Then decide which has at least 3 correct decimal places.

In [ ]:
# TODO: complete Exercise 1
true_val = math.sqrt(2)
candidates = [1.4, 1.41, 1.414]

for c in candidates:
    ae = absolute_error(true_val, c)
    re = relative_error(true_val, c)
    print(c, ae, re)

## 2) Machine Precision and Floating-Point Limits

In [ ]:
eps = 1.0
while 1.0 + eps > 1.0:
    eps /= 2
eps *= 2

print(f"Estimated machine epsilon: {eps:.20e}")
print(f"NumPy machine epsilon:     {np.finfo(float).eps:.20e}")

### Exercise 2
Modify the epsilon loop to estimate machine epsilon for `np.float32` instead of Python `float`.

Hint: use `np.float32(1.0)` and keep arithmetic in float32.

In [ ]:
# TODO: complete Exercise 2 (float32 epsilon)
eps32 = np.float32(1.0)
one32 = np.float32(1.0)

while np.float32(one32 + eps32) > one32:
    eps32 = np.float32(eps32 / np.float32(2.0))

eps32 = np.float32(eps32 * np.float32(2.0))
print("Estimated float32 epsilon:", eps32)
print("NumPy float32 epsilon:   ", np.finfo(np.float32).eps)

## §1.2 Floating-Point Representation: IEEE 754, Chopping vs Rounding

Every real number is stored as a **normalized floating-point** number:

$$x = \pm (1.b_1 b_2 \ldots b_{52}) \times 2^e$$

IEEE 754 double precision uses: **1 sign bit**, **11 exponent bits**, **52 mantissa bits**.

Two strategies for converting an infinite representation to finite precision:
- **Chopping**: discard all bits beyond the precision limit (biased low)
- **Rounding**: round to the nearest representable value (standard in Python/NumPy)

The difference matters: `0.1` is not exactly representable in binary — inspect what gets stored.

In [ ]:
import struct

def float_to_parts(x: float):
    bits = "".join(f"{b:08b}" for b in struct.pack(">d", x))
    sign, exp_bits, mantissa = bits[0], bits[1:12], bits[12:]
    exp_val = int(exp_bits, 2) - 1023
    return sign, exp_bits, exp_val, mantissa


print(f"{'value':>12}  {'sign':>4}  {'exponent':>10}  {'exp_dec':>7}  {'mantissa (first 20 bits)':>24}")
for v in [1.0, -1.0, 0.1, 1.0/3.0, math.pi]:
    s, eb, ev, m = float_to_parts(v)
    print(f"{v:>12.6g}    {s:>4}  {eb:>10}  {ev:>7}  {m[:20]:>24}")

# Demonstrate chopping vs rounding for 1/3 at 4 decimal digits
print()
true_third = 1.0 / 3.0
chopped = math.floor(true_third * 10000) / 10000   # truncate to 4 dec places
rounded = round(true_third, 4)
print(f"1/3 exact : {true_third:.15f}")
print(f"chopped   : {chopped:.15f}   abs error = {abs(true_third - chopped):.3e}")
print(f"rounded   : {rounded:.15f}   abs error = {abs(true_third - rounded):.3e}")

## Finite Precision Arithmetic

The exercises in Burden & Burden discuss decimal numbers with finite precision (e.g., 4-digit decimal arithmetic). To experiment with this, we create a `FinitePrecision` class that represents numbers constrained to a fixed number of decimal places. All arithmetic operations (+, −, ×, ÷) automatically round results to maintain the precision constraint.

This allows hands-on comparison of exact vs. finite-precision arithmetic.

In [36]:
import sys
sys.path.insert(0, '../ch02')
from finite_precision import FinitePrecision

# Example: Compare exact vs. 4-digit arithmetic
print("Demonstration: 3-digit precision arithmetic")
print("=" * 50)

x = FinitePrecision(1/30, precision=3)
y = FinitePrecision(1/30, precision=3)
z = x + y

print(f"x = 1/3 (3-digit) = {x.value}")
print(f"y = 1/3 (3-digit) = {y.value}")
print(f"x + y (3-digit) = {z.value}")
print(f"Exact: 1/3 + 1/3 = {1/3 + 1/3}")
print()

a = FinitePrecision(0.1, precision=4)
b = FinitePrecision(0.2, precision=4)
c = a * b

print(f"a = 0.1 (4-digit) = {a.value}")
print(f"b = 0.2 (4-digit) = {b.value}")
print(f"a * b (4-digit) = {c.value}")
print(f"Exact: 0.1 × 0.2 = {0.1 * 0.2}")

Demonstration: 3-digit precision arithmetic
x = 1/3 (3-digit) = 0.033
y = 1/3 (3-digit) = 0.033
x + y (3-digit) = 0.066
Exact: 1/3 + 1/3 = 0.6666666666666666

a = 0.1 (4-digit) = 0.1
b = 0.2 (4-digit) = 0.2
a * b (4-digit) = 0.02
Exact: 0.1 × 0.2 = 0.020000000000000004


In [ ]:
# Example: Quadratic roots with finite precision
print("\nExample: Quadratic roots (a, b, c are 4-digit)")
print("=" * 50)

# Standard form: ax² + bx + c = 0
a_fp = FinitePrecision(1, precision=4)
b_fp = FinitePrecision(-5.1, precision=4)
c_fp = FinitePrecision(1, precision=4)

discriminant_fp = b_fp * b_fp - 4 * a_fp * c_fp
print(f"a = {a_fp.value}, b = {b_fp.value}, c = {c_fp.value}")
print(f"Discriminant (4-digit) = {discriminant_fp.value}")
print(f"Exact discriminant = {(-5.1)**2 - 4*1*1}")

# We can mix FinitePrecision with regular numbers
sqrt_disc = (discriminant_fp.value) ** 0.5
root1 = (-b_fp + sqrt_disc) / (2 * a_fp)
root2 = (-b_fp - sqrt_disc) / (2 * a_fp)

print(f"Root 1 (4-digit) = {root1.value}")
print(f"Root 2 (4-digit) = {root2.value}")
print()

# Example: Summation with finite precision
print("Example: Sum of finite-precision numbers")
print("=" * 50)

terms = [FinitePrecision(0.1, precision=3) for _ in range(5)]
total = FinitePrecision(0, precision=3)

for i, term in enumerate(terms):
    total = total + term
    print(f"After term {i+1}: {total.value}")

print(f"Exact sum: {5 * 0.1}")
print(f"3-digit sum: {total.value}")
print(f"Difference: {abs(5 * 0.1 - total.value)}")

## §1.3a Rate of Convergence and Big-O Notation

A sequence of approximations $\{\alpha_n\}$ converges to $\alpha$ **at rate** $O(h^p)$ if:

$$|\alpha - \alpha_n| \leq C\,|h_n|^p \quad \text{for some constant } C > 0$$

**Big-O notation**: $f(h) = O(g(h))$ as $h \to 0$ means $|f(h)| \leq C|g(h)|$ for small $h$.

Classic example — derivative approximation:
- Forward difference: $f'(x) \approx \dfrac{f(x+h)-f(x)}{h}$ has error $O(h)$
- Centred difference: $f'(x) \approx \dfrac{f(x+h)-f(x-h)}{2h}$ has error $O(h^2)$

On a log-log plot, $O(h)$ appears as a line with slope 1; $O(h^2)$ as slope 2.

In [ ]:
f_func  = math.sin
df_true = math.cos
x0 = 1.0

h_vals = np.logspace(-1, -10, 100)
forward_err = np.array([abs((f_func(x0 + h) - f_func(x0)) / h - df_true(x0)) for h in h_vals])
centred_err = np.array([abs((f_func(x0 + h) - f_func(x0 - h)) / (2*h) - df_true(x0)) for h in h_vals])

plt.figure(figsize=(8, 4.5))
plt.loglog(h_vals, forward_err, label="forward diff  O(h)")
plt.loglog(h_vals, centred_err, label="centred diff  O(h\u00b2)")
plt.loglog(h_vals, h_vals,       "k:",  label="reference O(h)")
plt.loglog(h_vals, h_vals**2,    "k--", label="reference O(h\u00b2)")
plt.xlabel("h")
plt.ylabel("absolute error in f'(x)")
plt.title("Rate of Convergence: Forward vs Centred Difference")
plt.grid(True, which="both", ls=":")
plt.legend()
plt.show()

## §1.3b Conditioning: Well-Posed vs Ill-Conditioned Problems

A problem is **well-conditioned** if small changes in input produce proportionally small changes in output.
It is **ill-conditioned** if small input errors get amplified into large output errors.

The **condition number** of $f$ at $x$:

$$\kappa(x) = \left|\frac{x\, f'(x)}{f(x)}\right|$$

- $\kappa \approx 1$ -> well-conditioned
- $\kappa \gg 1$ -> ill-conditioned

An algorithm is **numerically stable** if errors it introduces are no worse than the inherent conditioning demands.

In [ ]:
def condition_number(f, df, x, tol=1e-12):
    fx = f(x)
    if abs(fx) < tol:
        return float("inf")
    return abs(x * df(x) / fx)


# Well-conditioned: sqrt(x), kappa = 0.5 everywhere
print("sqrt(x) - condition number (kappa = 0.5 always):")
for xv in [0.01, 0.1, 1.0, 100.0]:
    k = condition_number(math.sqrt, lambda x: 0.5 / math.sqrt(x), xv)
    print(f"  x = {xv:7.2f}   kappa = {k:.4f}")

# Ill-conditioned near x=0: f(x) = x - sin(x), outcome amplified
print()
print("x - sin(x) near x=0 (ill-conditioned):")
def f_ill(x):  return x - math.sin(x)
def df_ill(x): return 1 - math.cos(x)
for xv in [1.0, 0.1, 0.01, 0.001]:
    k = condition_number(f_ill, df_ill, xv)
    print(f"  x = {xv:.3f}   kappa = {k:.2e}")

## 3) Catastrophic Cancellation

For small $x$, `1 - cos(x)` loses significant digits.
A numerically stable equivalent is `2 * sin(x/2)**2`.

In [ ]:
x_values = np.logspace(-16, -1, 80)

naive = 1 - np.cos(x_values)
stable = 2 * np.sin(x_values / 2) ** 2

# Decimal-based short Taylor model as reference for very small x
def decimal_reference(x: float) -> Decimal:
    xd = Decimal(str(x))
    x2 = xd * xd
    return x2 / 2 - (x2 * x2) / 24 + (x2 * x2 * x2) / 720

ref = np.array([float(decimal_reference(x)) for x in x_values])
naive_rel = np.abs((naive - ref) / ref)
stable_rel = np.abs((stable - ref) / ref)

plt.figure(figsize=(8, 4.5))
plt.loglog(x_values, naive_rel, label="naive: 1 - cos(x)")
plt.loglog(x_values, stable_rel, label="stable: 2 sin^2(x/2)")
plt.xlabel("x")
plt.ylabel("relative error")
plt.title("Cancellation Error for Small x")
plt.grid(True, which="both", ls=":")
plt.legend()
plt.show()

### Exercise 3
Find the smallest `x = 10^{-k}` (integer `k`) where the naive formula evaluates to exactly `0.0` in double precision.

In [ ]:
# TODO: complete Exercise 3
for k in range(1, 30):
    x = 10.0 ** (-k)
    y = 1.0 - math.cos(x)
    if y == 0.0:
        print(f"First k where naive expression is 0.0: k={k}, x={x:.1e}")
        break

### Exercise 3B: Quadratic Formula Stability
For `a*x^2 + b*x + c = 0` with `b^2 >> 4ac`, one root from `(-b + sqrt(b^2 - 4ac)) / (2a)` can suffer cancellation.

Use the stable strategy:
- compute `x1 = (-b - sign(b)*sqrt(b^2-4ac)) / (2a)`
- compute the second root from `x2 = c / (a*x1)`

Try this with very small `a` and `c` and compare naive vs stable roots.

In [37]:
# TODO: complete Exercise 3B
def quadratic_naive(a, b, c):
    disc = b*b - 4*a*c
    s = math.sqrt(disc)
    x_plus = (-b + s) / (2*a)
    x_minus = (-b - s) / (2*a)
    return x_plus, x_minus


def quadratic_stable(a, b, c):
    disc = b*b - 4*a*c
    s = math.sqrt(disc)
    x1 = (-b - math.copysign(s, b)) / (2*a)
    x2 = c / (a * x1)
    return x1, x2


a, b, c = 1e-12, 1.0, 1e-12
n1, n2 = quadratic_naive(a, b, c)
s1, s2 = quadratic_stable(a, b, c)

print("naive roots:", n1, n2)
print("stable roots:", s1, s2)
print("residuals stable:", a*s1*s1 + b*s1 + c, a*s2*s2 + b*s2 + c)

naive roots: 0.0 -1000000000000.0
stable roots: -1000000000000.0 -1e-12
residuals stable: 1e-12 0.0


### Exercise 3C: Stable Evaluation of `pi/2 - atan(x)`
For large `x`, directly computing `pi/2 - atan(x)` loses precision.

Use the identity `pi/2 - atan(x) = atan(1/x)` for `x > 0` as a stable approximation.

Compare both formulas for `x = 10^k`, `k = 1, ..., 16`, and inspect the relative difference.

In [ ]:
# TODO: complete Exercise 3C
print(f"{'x':>12} {'direct':>18} {'stable':>18} {'rel_diff':>14}")
for k in range(1, 17):
    x = 10.0 ** k
    direct = math.pi / 2 - math.atan(x)
    stable = math.atan(1.0 / x)
    rel_diff = abs(direct - stable) / abs(stable)
    print(f"{x:12.1e} {direct:18.10e} {stable:18.10e} {rel_diff:14.3e}")

## 4) Linear vs Exponential Error Growth

Consider the recurrence
`y_n = y_{n-1} + y_{n-2}`, with `y_0 = 1` and `y_1 = -1/phi`, where `phi = (1+sqrt(5))/2`.

In exact arithmetic, `y_n = (-1/phi)^n`, so the sequence decays in magnitude.
In floating-point arithmetic, tiny perturbations introduce a `phi^n` mode, and error can grow exponentially.

Below, we compare:
- exact decay `|(-1/phi)^n|` (straight line on semilog scale: exponential in `n`)
- observed numerical absolute error (often bends upward due to exponential amplification).

In [ ]:
phi = (1 + math.sqrt(5.0)) / 2.0
n_max = 90

# Numerical recurrence in float arithmetic
y_num = np.empty(n_max + 1, dtype=float)
y_num[0] = 1.0
y_num[1] = -1.0 / phi
for n in range(2, n_max + 1):
    y_num[n] = y_num[n - 1] + y_num[n - 2]

# Exact closed-form sequence
n_idx = np.arange(n_max + 1)
y_exact = (-1.0 / phi) ** n_idx

abs_exact = np.abs(y_exact)
abs_err = np.abs(y_num - y_exact)

# Estimate early linear growth and late exponential growth in log scale
n_split = 35
coef_lin = np.polyfit(n_idx[2:n_split], np.log(abs_err[2:n_split] + 1e-300), 1)
coef_exp = np.polyfit(n_idx[n_split:], np.log(abs_err[n_split:] + 1e-300), 1)

print(f"Early log-error slope (near linear trend): {coef_lin[0]:.4f}")
print(f"Late log-error slope (exponential trend):   {coef_exp[0]:.4f}")

# Create figure with two subplots
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))

# Top plot: Series values (absolute values on log scale)
ax1.semilogy(n_idx, abs_exact, "k--", linewidth=2, label="|Exact: (-1/φ)^n|")
ax1.semilogy(n_idx, np.abs(y_num), "b-", linewidth=1.5, label="|Numerical: recurrence relation|")
ax1.axvline(n_split, color="gray", ls=":", lw=1.5, alpha=0.7, label="transition index")
ax1.set_xlabel("n")
ax1.set_ylabel("|y_n| (log scale)")
ax1.set_title("Fibonacci Recurrence: Series Values")
ax1.grid(True, which="both", ls=":", alpha=0.5)
ax1.legend()

# Bottom plot: Error analysis (starting from n=4)
ax2.semilogy(n_idx[4:], abs_exact[4:], "k--", label="|exact| = |(-1/φ)^n|")
ax2.semilogy(n_idx[4:], abs_err[4:] + 1e-300, "r", label="|numerical error|")
ax2.axvline(n_split, color="gray", ls=":", lw=1.5, alpha=0.7, label="transition index")
ax2.set_xlabel("n")
ax2.set_ylabel("magnitude (log scale)")
ax2.set_title("Decay of Signal vs Growth of Round-off Error")
ax2.grid(True, which="both", ls=":", alpha=0.5)
ax2.legend()

plt.tight_layout()
plt.show()

## 5) Summation Order and Numerical Stability

Floating-point addition is not associative, so summation order can matter.

In [ ]:
# Many tiny terms and very few huge terms: stress test for summation stability
small_count = 2_000_000
small_value = 1e-8
large_count = 100
large_value = 1e8

# Ratio of tiny to huge terms is intentionally very large
print(f"small terms: {small_count:,}, large terms: {2*large_count:,}")

# True sum: small_count * small_value = 0.02
data = [large_value] * large_count + [small_value] * small_count + [-large_value] * large_count

rng_state = random.Random(42)
data_random = data[:]
rng_state.shuffle(data_random)

def plain_sum(values):
    s = 0.0
    for v in values:
        s += v
    return s


def kahan_sum(values):
    s = 0.0
    c = 0.0
    for v in values:
        y = v - c
        t = s + y
        c = (t - s) - y
        s = t
    return s

true_sum = float(Decimal(str(small_value)) * Decimal(small_count))

forward    = plain_sum(data)
backward   = plain_sum(list(reversed(data)))
rand_order = plain_sum(data_random)
sorted_mag = plain_sum(sorted(data, key=abs))
kahan      = kahan_sum(data)

print(f"reference:             {true_sum:.12f}")
print(f"plain (big/small/big): {forward:.12f}")
print(f"plain (reversed):      {backward:.12f}")
print(f"plain (random order):  {rand_order:.12f}")
print(f"plain (sorted by |x|): {sorted_mag:.12f}")
print(f"kahan:                 {kahan:.12f}")

### Exercise 4
Construct your own sequence where naive summation gives a poor result but Kahan summation improves it.

Try combining one very large term with many tiny terms.

In [11]:
# TODO: complete Exercise 4
my_data = [1e16] + [1.0] * 100000 + [-1e16]
print("plain:", plain_sum(my_data))
print("kahan:", kahan_sum(my_data))

plain: 0.0
kahan: 100000.0


## 6) Mini Project

Write a short report (markdown cell) using Chapter 1 ideas:
1. Pick one unstable formula and one stable reformulation.
2. Compare errors on a meaningful input range.
3. Explain which method you would deploy and why.